In [12]:
import os, random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.metrics import confusion_matrix, classification_report

In [2]:
import os,random,shutil
import tensorflow as tf
SEED = 42
random.seed(SEED)

In [3]:
def split_dataset(src_dir, out_dir, classes, ratios=(0.8, 0.1, 0.1)):
    random.seed(SEED)  
    for cls in classes:
        src = os.path.join(src_dir, cls)
        files = os.listdir(src)
        random.shuffle(files)         

        n = len(files)
        n_train = int(n * ratios[0])
        n_val   = int(n * ratios[1])

        splits = {
            "train": files[:n_train],
            "val":   files[n_train : n_train + n_val],
            "test":  files[n_train + n_val :],
        }

        for split_name, split_files in splits.items():
            dst = os.path.join(out_dir, split_name, cls)
            os.makedirs(dst, exist_ok=True)   
            for f in split_files:
                shutil.copy(os.path.join(src, f), os.path.join(dst, f))

        print(f"{cls}: train={len(splits['train'])}, "
              f"val={len(splits['val'])}, test={len(splits['test'])}")

In [4]:
split_dataset(
    src_dir="PetImages",
    out_dir="data_split",
    classes=["Cat", "Dog"],
)

Cat: train=9999, val=1249, test=1251
Dog: train=9999, val=1249, test=1251


In [5]:
for split in ["train", "val", "test"]:
    for cls in ["Cat", "Dog"]:
        p = os.path.join("data_split", split, cls)
        n = len(os.listdir(p))
        print(f"{split:5s} {cls}: {n}")

train Cat: 9999
train Dog: 9999
val   Cat: 1249
val   Dog: 1249
test  Cat: 1251
test  Dog: 1251


In [6]:
def names(folder):
    return set(os.listdir(folder))

cat_train = names("data_split/train/Cat")
cat_val   = names("data_split/val/Cat")
cat_test  = names("data_split/test/Cat")

print("cat overlap train∩val:", cat_train & cat_val)
print("cat overlap train∩test:", cat_train & cat_test)
print("cat overlap val∩test:", cat_val & cat_test)

cat overlap train∩val: set()
cat overlap train∩test: set()
cat overlap val∩test: set()


In [7]:
IMG_SIZE = (180,180)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

In [8]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    "data_split/train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=True,
    seed=SEED,
)

Found 19998 files belonging to 2 classes.


In [9]:
val_ds = tf.keras.utils.image_dataset_from_directory(
    "data_split/val",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=False,
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    "data_split/test",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=False,
)

Found 2498 files belonging to 2 classes.
Found 2502 files belonging to 2 classes.


In [10]:
for images, labels in train_ds.take(1):
    print("images shape:", images.shape)
    print("labels shape:", labels.shape)
    print("image dtype :", images.dtype)
    print("pixel min/max:", images.numpy().min(), images.numpy().max())
    print("label values:", labels.numpy().ravel()[:8])

images shape: (32, 180, 180, 3)
labels shape: (32, 1)
image dtype : <dtype: 'float32'>
pixel min/max: 0.0 255.0
label values: [0. 1. 0. 1. 1. 0. 0. 0.]


In [13]:
normalization_layer = layers.Rescaling(1./255)

train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
val_ds   = val_ds.map(lambda x, y: (normalization_layer(x), y))
test_ds  = test_ds.map(lambda x, y: (normalization_layer(x), y))

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds   = val_ds.prefetch(AUTOTUNE)
test_ds  = test_ds.prefetch(AUTOTUNE)

In [14]:
for images, labels in train_ds.take(1):
    print("pixel min/max after normalize:", images.numpy().min(), images.numpy().max())

pixel min/max after normalize: 0.0 1.0
